In [2]:
!pip3 install xgboost

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.9/253.9 MB 11.0 MB/s eta 0:00:0000:0100:01


In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from abc import ABC, abstractmethod
from pathlib import Path

from cashflow import ScorableModelTemplate
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, GridSearchCV
from sklearn.metrics import roc_auc_score

from datetime import timedelta

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler


import warnings
warnings.filterwarnings("ignore")



In [2]:
def process_inputs(raw_consumer_file: str, raw_transactions_file: str):
    """Input argument will vary. See you competition's template.

    :param raw_files: list of file path strings, depends on competition
    :return: anything needed for you model to make predictions, e.g. features or processed data
    """
    # Read in files
    df_consumer = pd.read_parquet(raw_consumer_file)
    df_transactions = pd.read_parquet(raw_transactions_file)

    # Merge and filter to get only valid data
    merged_df = df_transactions.merge(df_consumer, on = "masked_consumer_id")
    #filtered_df = merged_df[merged_df["posted_date"]< merged_df["evaluation_date"]]
    filtered_df = merged_df
    # Clean evaluation_date and create evaluation_day for day of week of evaluation_date
    filtered_df['evaluation_date'] = pd.to_datetime(filtered_df['evaluation_date'])
    filtered_df['evaluation_day'] = filtered_df['evaluation_date'].dt.dayofweek

    # Clean posted_date and create posted_day for day of week of posted_date
    filtered_df['posted_date'] = pd.to_datetime(filtered_df['posted_date'])
    filtered_df['posted_day'] = filtered_df['posted_date'].dt.dayofweek

    # Get loan category from masked_consumer_id
    filtered_df['loan_category'] = filtered_df['masked_consumer_id'].str[2].astype(int)
    
    return filtered_df
    
df_consumers = process_inputs("~/private/DSC 190/mlc/consumers_test_data.parquet", "~/private/DSC 190/mlc/transactions_test_data.parquet")

# TRAINING DATA
df_train = process_inputs(
    "~/private/DSC 190/mlc/test_data/consumer_data.parquet",
    "~/private/DSC 190/mlc/test_data/transactions.parquet"
)
df_train = df_train[df_train['loan_category'] == 3]

### TRENDS



In [57]:
import ast
from sklearn.linear_model import LinearRegression

transaction_grouped = pd.read_csv("~/private/DSC 190/mlc" + '/cashflow-data/features.csv')
for col in transaction_grouped.columns[1:]:
    transaction_grouped[col] = transaction_grouped[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)


def compute_trend(series):
    if not isinstance(series, list):
        return 0  # fallback if data malformed

    series = np.array(series)
    nonzero_idx = np.nonzero(series)[0]
    nonzero_vals = series[nonzero_idx]

    if len(nonzero_vals) <= 2:
        return 0

    X = nonzero_idx.reshape(-1, 1)  # time indices
    y = nonzero_vals
    model = LinearRegression().fit(X, y)
    return model.coef_[0]  # slope

# Apply to each time series column
trend_df = pd.DataFrame()
trend_df['masked_consumer_id'] = transaction_grouped['masked_consumer_id']

# All time-series columns
series_cols = transaction_grouped.columns.drop('masked_consumer_id')

for col in series_cols:
    trend_df[col + '_trend'] = transaction_grouped[col].apply(compute_trend)

trend_df.head()

,masked_consumer_id,net_week_series_trend,count_week_series_trend,inflow_week_series_trend,outflow_week_series_trend,week_series_category_0_trend,week_series_category_1_trend,week_series_category_2_trend,week_series_category_3_trend,week_series_category_4_trend,...,week_series_category_26_trend,week_series_category_27_trend,week_series_category_28_trend,week_series_category_29_trend,week_series_category_30_trend,week_series_category_31_trend,week_series_category_32_trend,week_series_category_33_trend,week_series_category_34_trend,week_series_category_35_trend
0,C03100001,2.532763,0.080820,-3.679276,1.707118,0.000000,8.754895,-169.171644,10.149108,-4.871593,...,0.635384,0.217971,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,C03100002,10.013128,2.000000,75.295939,-65.282811,0.000000,44.093005,17.326267,0.000000,0.000000,...,-2.201557,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,C03100003,51.292377,0.095588,375.922794,-324.630417,-253.383459,65.739889,-49.195471,-20.221083,0.000000,...,-1.943983,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,C03100004,-2.696664,0.695565,29.113796,-31.810460,0.000000,2.145596,-56.506849,0.000000,-2.849789,...,0.000000,-0.502318,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,C03100005,-0.274092,0.072776,-4.213670,2.299438,19.691781,-2.929210,-2.091014,0.000000,0.000000,...,-0.052188,0.000000,-0.733599,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [58]:
len(trend_df)

4000

### FEATURES

#### category
- 0,SELF_TRANSFER: 'amount_mean', amount_median, 'amount_std', 'amount_max', 'amount_75p', 'balance_mean', 'balance_median', 'balance_min', 'balance_max', 'balance_25p', 'balance_75p', 'outflow_sum', 'freq_per_month', 'freq_per_week'
- 1,EXTERNAL_TRANSFER: 'num_transact', 'num_outflow', 'num_inflow', 'amount_median', 'amount_std', 'amount_min', 'amount_max', 'amount_25p', 'balance_mean', 'balance_median', 'balance_std', 'balance_min', 'balance_max', 'balance_25p', 'balance_75p', 'days_between_mean', 'days_between_std', 'inflow_sum', 'outflow_sum', 'txn_count', 'freq_per_weekday'
- 2,DEPOSIT: 'amount_std', 'balance_mean', 'balance_median', 'balance_std', 'balance_min', 'balance_max', 'balance_25p', 'balance_75p', 'days_between_std', 'days_between_skew', 'inflow_sum', 'net_flow_sum', 'freq_per_month', 'freq_per_week', 'amt_per_weekday'
- 3,PAYCHECK: 'num_transact', 'amount_mean', 'amount_median', 'amount_max', 'amount_25p', 'amount_75p', 'balance_mean', 'balance_median', 'balance_std', 'balance_min', 'balance_max', 'balance_25p', 'balance_75p', 'days_between_std', 'inflow_sum', 'net_flow_sum', 'txn_count', 'freq_per_month', 'freq_per_weekday', 'amt_per_weekday', 'amt_per_month', 'amt_per_week'
- 4,MISCELLANEOUS: 'amount_mean', 'amount_median', 'amount_std', 'amount_75p', 'balance_mean', 'balance_median', 'balance_std', 'balance_min', 'balance_max', 'balance_25p', 'balance_75p', 'days_between_mean', 'inflow_sum', 'net_flow_sum', 'amt_per_weekday', 'amt_per_month', 'amt_per_week'
- 5,PAYCHECK_PLACEHOLDER: 'amount_mean', 'amount_median', 'amount_std', 'amount_max', 'amount_25p', 'amount_75p', 'balance_mean', 'balance_median', 'balance_std', 'balance_min', 'balance_max', 'balance_25p', 'balance_75p', 'days_between_mean', 'days_between_std', 'inflow_sum', 'net_flow_sum', 'amt_per_weekday', 'amt_per_month', 'amt_per_week'
- 6,REFUND: 'num_transact', 'balance_mean', 'balance_median', 'balance_min', 'balance_max', 'balance_25p', 'balance_75p', 'txn_count', 'freq_per_month', 'freq_per_weekday', 'freq_per_week'
- 7,INVESTMENT_INCOME: 'amount_mean', 'amount_median', 'amount_std', 'amount_min', 'amount_max', 'amount_25p', 'amount_75p', 'amount_skew', 'balance_mean', 'balance_median', 'balance_std', 'balance_min', 'balance_max', 'balance_25p', 'balance_75p', 'inflow_sum', 'net_flow_sum', 'freq_per_month', 'freq_per_weekday', 'amt_per_weekday', 'amt_per_month', 'amt_per_week'
- 8,OTHER_BENEFITS: nothing
- 9,UNEMPLOYMENT_BENEFITS: 'balance_mean', 'balance_median', 'balance_std', 'balance_min', 'balance_max', 'balance_25p', 'balance_75p', 'days_between_std', 'days_between_skew'
- 10,SMALL_DOLLAR_ADVANCE: 'amount_std', 'amount_max', 'balance_mean', 'balance_median', 'balance_std', 'balance_min', 'balance_max', 'balance_25p', 'balance_75p', 'inflow_sum', 'net_flow_sum', 'amt_per_weekday'
- 11,TAX: 'balance_mean', 'balance_median', 'balance_std', 'balance_min', 'balance_max', 'balance_25p', 'balance_75p'
- 12,LOAN: 'num_transact', 'num_outflow', 'amount_mean', 'amount_median', 'amount_std', 'amount_min', 'amount_max', 'amount_25p', 'balance_mean', 'balance_median', 'balance_std', 'balance_min', 'balance_max', 'balance_25p', 'balance_75p', 'days_between_mean', 'days_between_std', 'outflow_sum', 'net_flow_sum', 'txn_count', 'freq_per_weekday', 'amt_per_weekday', 'amt_per_month', 'amt_per_week'
- 13,INSURANCE: 'num_transact', 'num_outflow', 'amount_median', 'amount_25p', 'balance_mean', 'balance_median', 'balance_std', 'balance_min', 'balance_max', 'balance_25p', 'balance_75p', 'inflow_sum', 'txn_count', 'freq_per_month', 'freq_per_weekday'
- 14,FOOD_AND_BEVERAGES: 'num_transact', 'num_outflow', 'amount_std', 'amount_min', 'amount_75p', 'amount_skew', 'balance_mean', 'balance_median', 'balance_std', 'balance_min', 'balance_max', 'balance_25p', 'balance_75p', 'days_between_mean', 'days_between_std', 'outflow_sum', 'net_flow_sum', 'txn_count', 'freq_per_month', 'freq_per_weekday', 'amt_per_weekday', 'amt_per_month'
- 15,UNCATEGORIZED: 'num_transact', 'num_outflow', 'amount_mean', 'amount_median', 'amount_std', 'amount_max', 'amount_25p', 'amount_75p', 'amount_skew', 'balance_mean', 'balance_median', 'balance_std', 'balance_min', 'balance_max', 'balance_25p', 'balance_75p', 'days_between_mean', 'days_between_std', 'outflow_sum', 'net_flow_sum', 'txn_count', 'freq_per_month', 'freq_per_weekday', 'amt_per_weekday', 'amt_per_month', 'amt_per_week'
- 16,GENERAL_MERCHANDISE: 'num_transact', 'num_outflow', 'amount_min', 'amount_skew', 'balance_mean', 'balance_median', 'balance_std', 'balance_min', 'balance_max', 'balance_25p', 'balance_75p', 'days_between_mean', 'days_between_std', 'outflow_sum', 'net_flow_sum', 'txn_count', 'freq_per_weekday', 'amt_per_weekday', 'amt_per_month'
- 17,AUTOMOTIVE: 'amount_mean', 'amount_median', 'amount_std', 'amount_min', 'amount_25p', 'amount_skew', 'balance_mean', 'balance_median', 'balance_std', 'balance_min', 'balance_max', 'balance_25p', 'balance_75p', 'days_between_mean', 'days_between_std', 'outflow_sum', 'net_flow_sum', 'freq_per_weekday', 'amt_per_weekday', 'amt_per_month', 'amt_per_week'
- 18,GROCERIES: 'num_transact', 'num_outflow', 'amount_75p', 'balance_mean', 'balance_median', 'balance_std', 'balance_min', 'balance_max', 'balance_25p', 'balance_75p', 'days_between_std', 'outflow_sum', 'net_flow_sum', 'txn_count', 'freq_per_weekday', 'amt_per_weekday', 'amt_per_month'
- 19,ATM_CASH: 
- 20,ENTERTAINMENT: 
- 21,TRAVEL: 
- 22,ESSENTIAL_SERVICES: 
- 23,ACCOUNT_FEES: 
- 24,HOME_IMPROVEMENT: 
- 25,OVERDRAFT: 
- 26,CREDIT_CARD_PAYMENT: 
- 27,HEALTHCARE_MEDICAL: 
- 28,PETS: 
- 29,EDUCATION: 
- 30,GIFTS_DONATIONS: 
- 31,BILLS_UTILITIES: 
- 32,MORTGAGE: 
- 33,CHILD_DEPENDENTS: 
- 34,RENT: 
- 35,BNPL: 
- 36,AUTO_LOAN: 

In [38]:
f"{1}_x"

'1_x'

In [54]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind, skew

def analyze_detailed_time_series(df_train, i):
    df = df_train.copy()

    # Ensure datetime
    df['posted_date'] = pd.to_datetime(df['posted_date'])
    df['evaluation_date'] = pd.to_datetime(df['evaluation_date'])

    # Derived time columns
    df['month'] = df['posted_date'].dt.to_period('M')
    df['weekday'] = df['posted_date'].dt.dayofweek
    df['week'] = df['posted_date'].dt.isocalendar().week
    df['year_week'] = df['posted_date'].dt.strftime('%Y-%U')
    df['days_between'] = (df['evaluation_date'] - df['posted_date']).dt.days

    # Inflow/outflow
    df['inflow'] = df['amount'].apply(lambda x: x if x > 0 else 0)
    df['outflow'] = df['amount'].apply(lambda x: -x if x < 0 else 0)
    df['net_flow'] = df['inflow'] - df['outflow']

    # Group by user and target
    grouped = df.groupby(['masked_consumer_id', 'FPF_TARGET'])
    # Main aggregations with distributional statistics
    def agg_stats(g):
        return pd.Series({
            f'{i}_num_transact': sum(g['amount'] != -10000102010),
            f'{i}_num_outflow': sum(g['amount'] < 0),
            f'{i}_num_inflow': sum(g['amount'] > 0),
            f'{i}_amount_mean': g['amount'].mean(),
            f'{i}_amount_median': g['amount'].median(),
            f'{i}_amount_std': g['amount'].std(),
            f'{i}_amount_min': g['amount'].min(),
            f'{i}_amount_max': g['amount'].max(),
            f'{i}_amount_25p': np.percentile(g['amount'], 25),
            f'{i}_amount_75p': np.percentile(g['amount'], 75),
            f'{i}_amount_skew': skew(g['amount']),
            f'{i}_balance_mean': g['total_balance'].mean(),
            f'{i}_balance_median': g['total_balance'].median(),
            f'{i}_balance_std': g['total_balance'].std(),
            f'{i}_balance_min': g['total_balance'].min(),
            f'{i}_balance_max': g['total_balance'].max(),
            f'{i}_balance_25p': np.percentile(g['total_balance'], 25),
            f'{i}_balance_75p': np.percentile(g['total_balance'], 75),
            f'{i}_balance_skew': skew(g['total_balance']),
            f'{i}_days_between_mean': g['days_between'].mean(),
            f'{i}_days_between_std': g['days_between'].std(),
            f'{i}_days_between_skew': skew(g['days_between']),
            f'{i}_inflow_sum': g['inflow'].sum(),
            f'{i}_outflow_sum': g['outflow'].sum(),
            f'{i}_net_flow_sum': g['net_flow'].sum(),
            f'{i}_txn_count': len(g)
        })

    stats_agg = grouped.apply(agg_stats).reset_index()

    # Frequency and amount patterns
    def time_freq(g):
        freq_month = g['month'].value_counts().mean()
        freq_weekday = g['weekday'].value_counts().mean()
        freq_week = g['year_week'].value_counts().mean()
        amt_weekday = g.groupby('weekday')['amount'].sum().mean()
        amt_month = g.groupby('month')['amount'].sum().mean()
        amt_week = g.groupby('year_week')['amount'].sum().mean()
        return pd.Series({
            f'{i}_freq_per_month': freq_month,
            f'{i}_freq_per_weekday': freq_weekday,
            f'{i}_freq_per_week': freq_week,
            f'{i}_amt_per_weekday': amt_weekday,
            f'{i}_amt_per_month': amt_month,
            f'{i}_amt_per_week': amt_week
        })

    time_agg = grouped.apply(time_freq).reset_index()

    # Combine
    features = pd.merge(stats_agg, time_agg, on=['masked_consumer_id', 'FPF_TARGET'])
    
    # Split groups
    g0 = features[features['FPF_TARGET'] == 0].drop(columns=['masked_consumer_id', 'FPF_TARGET'])
    g1 = features[features['FPF_TARGET'] == 1].drop(columns=['masked_consumer_id', 'FPF_TARGET'])

    # Summary stats
    summary = pd.concat([
        g0.describe(percentiles=[.25, .5, .75]).T.rename(columns={
            'mean': 'mean_0', 'std': 'std_0', 'min': 'min_0', 'max': 'max_0',
            '25%': '25p_0', '50%': 'median_0', '75%': '75p_0'
        })[['mean_0', 'std_0', 'min_0', '25p_0', 'median_0', '75p_0', 'max_0']],
        g1.describe(percentiles=[.25, .5, .75]).T.rename(columns={
            'mean': 'mean_1', 'std': 'std_1', 'min': 'min_1', 'max': 'max_1',
            '25%': '25p_1', '50%': 'median_1', '75%': '75p_1'
        })[['mean_1', 'std_1', 'min_1', '25p_1', 'median_1', '75p_1', 'max_1']]
    ], axis=1)

    # T-tests
    ttest_results = {}
    for col in g0.columns:
        stat, pval = ttest_ind(g0[col].dropna(), g1[col].dropna(), equal_var=False)
        ttest_results[col] = {'t_stat': stat, 'p_value': pval}

    return features, summary, ttest_results




features, summary_stats, test_results = analyze_detailed_time_series(df_train[df_train.category == 0], 0)

# Show statistically significant differences (p < 0.05)
#significant_diffs = {k: v for k, v in test_results.items() if v['p_value'] < 0.05}
#print([k for k,v in significant_diffs.items()])

In [56]:
fin_features

,masked_consumer_id,FPF_TARGET,0_amount_mean,0_amount_std,0_amount_max,0_amount_75p,0_balance_mean,0_balance_median,0_balance_min,0_balance_max,...,34_balance_min,34_balance_max,34_balance_25p,34_balance_75p,35_amount_min,35_balance_std,35_freq_per_month,35_freq_per_week,35_amt_per_weekday,35_amt_per_month
0,C03100001,0.0,-17.735000,0.346482,-17.49,-17.6125,359.64,359.64,359.64,359.64,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,C03100002,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,C03100003,0.0,1801.666667,2774.032865,5000.00,2677.5000,0.21,0.21,0.21,0.21,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,C03100004,0.0,86.000000,NaN,86.00,86.0000,1054.65,1054.65,1054.65,1054.65,...,NaN,NaN,NaN,NaN,-26.56,NaN,1.0,1.0,-26.56,-26.56
4,C03100005,0.0,-312.500000,198.588771,-25.00,-175.0000,59.43,59.43,59.43,59.43,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3995,C03104996,0.0,311.951935,331.813870,1348.00,375.0000,709.34,709.34,709.34,709.34,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3996,C03104997,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3997,C03104998,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3998,C03104999,0.0,1191.933333,219.012853,1321.79,1318.3650,308.00,308.00,308.00,308.00,...,308.0,308.0,308.0,308.0,NaN,NaN,NaN,NaN,NaN,NaN


In [55]:
fin_table = df_train.groupby('masked_consumer_id')['FPF_TARGET'].first().reset_index()
features, summary_stats, test_results = analyze_detailed_time_series(df_train[df_train.category == 0], 0)
significant_diffs = {k: v for k, v in test_results.items() if v['p_value'] < 0.05}
sig_feat = [k for k,v in significant_diffs.items()]
sig_feat.append('masked_consumer_id')

fin_features = pd.merge(fin_table, features[sig_feat], on = 'masked_consumer_id', how = 'left')
print(len(fin_features))
for i in range(1,37):
    print(i)
    features, summary_stats, test_results = analyze_detailed_time_series(df_train[df_train.category == i], i)
    significant_diffs = {k: v for k, v in test_results.items() if v['p_value'] < 0.05}
    sig_feat = [k for k,v in significant_diffs.items()]
    sig_feat.append('masked_consumer_id')
    fin_features = pd.merge(fin_features, features[sig_feat], on = 'masked_consumer_id', how = 'left')
    print(len(fin_features))

4000
1
4000
2
4000
3
4000
4
4000
5
4000
6
4000
7
4000
8
4000
9
4000
10
4000
11
4000
12
4000
13
4000
14
4000
15
4000
16
4000
17
4000
18
4000
19
4000
20
4000
21
4000
22
4000
23
4000
24
4000
25
4000
26
4000
27
4000
28
4000
29
4000
30
4000
31
4000
32
4000
33
4000
34
4000
35
4000
36


ValueError: cannot insert FPF_TARGET, already exists

In [59]:
all_features = pd.merge(trend_df, fin_features, on = 'masked_consumer_id')

In [68]:
# Configurable version: Toggle features via dictionary
def per_user_features_config(user_df):
    result = {}
    
    result['n_records'] = len(user_df)
    result['n_unique_posted_days'] = user_df['posted_date'].nunique()
        
    posted_counts = user_df.groupby('posted_date').size()
    result['multi_entry_days'] = (posted_counts > 1).sum()

    result['active_days'] = (user_df['posted_date'].max() - user_df['posted_date'].min()).days + 1
    post_dates = user_df['posted_date'].sort_values().unique()
    if len(post_dates) > 1:
        time_deltas = np.diff(post_dates) / np.timedelta64(1, 'D')
        result['mean_gap_days'] = np.mean(time_deltas)
        result['std_gap_days'] = np.std(time_deltas)
    else:
        result['mean_gap_days'] = 0
        result['std_gap_days'] = 0

    user_df['day_of_week'] = user_df['posted_date'].dt.dayofweek
    dow_spending = user_df.groupby('day_of_week')['amount'].sum()
    for dow in range(7):
        result[f'dow_{dow}_total_spent'] = dow_spending.get(dow, 0)
    

    for col in ['amount', 'total_balance']:
        vals = user_df[col].values
        result[f'{col}_mean'] = np.mean(vals)
        result[f'{col}_std'] = np.std(vals)
        result[f'{col}_skew'] = pd.Series(vals).skew()
        result[f'{col}_min'] = np.min(vals)
        result[f'{col}_max'] = np.max(vals)
        result[f'{col}_pos_ratio'] = (vals > 0).mean()

    return pd.Series(result)


features_df = df_train.groupby('masked_consumer_id').apply(per_user_features_config).reset_index()

all_features_test = pd.merge(all_features, features_df, on = 'masked_consumer_id')

In [84]:
features_df

,masked_consumer_id,n_records,n_unique_posted_days,multi_entry_days,active_days,mean_gap_days,std_gap_days,dow_0_total_spent,dow_1_total_spent,dow_2_total_spent,...,amount_skew,amount_min,amount_max,amount_pos_ratio,total_balance_mean,total_balance_std,total_balance_skew,total_balance_min,total_balance_max,total_balance_pos_ratio
0,C03100001,2511.0,368.0,354.0,535.0,1.455041,0.881243,-42149.46,-11659.15,-1578.09,...,14.509016,-2800.00,15665.09,0.042613,359.64,1.136868e-13,0.0,359.64,359.64,1.0
1,C03100002,1252.0,133.0,129.0,197.0,1.484848,0.891754,-11862.39,1322.98,-361.53,...,1.320493,-3003.94,3000.00,0.253195,1143.71,0.000000e+00,0.0,1143.71,1143.71,1.0
2,C03100003,445.0,104.0,93.0,120.0,1.155340,0.516848,21093.80,-3004.72,-13205.08,...,0.092366,-30367.10,30367.10,0.175281,0.21,2.775558e-17,0.0,0.21,0.21,1.0
3,C03100004,1326.0,160.0,157.0,228.0,1.427673,0.864820,-1940.55,1196.45,-2608.91,...,8.959570,-1380.00,4237.94,0.092760,1054.65,2.273737e-13,0.0,1054.65,1054.65,1.0
4,C03100005,772.0,299.0,197.0,480.0,1.607383,0.946672,-1284.88,-1410.02,-872.77,...,1.325537,-500.00,600.00,0.173575,59.43,7.105427e-15,0.0,59.43,59.43,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3995,C03104996,1675.0,339.0,296.0,526.0,1.553254,0.947656,-17135.57,-14376.23,32254.67,...,3.173087,-3081.70,3555.00,0.081791,709.34,0.000000e+00,0.0,709.34,709.34,1.0
3996,C03104997,82.0,18.0,15.0,30.0,1.705882,1.225451,-1206.78,-257.03,-330.80,...,6.285249,-698.56,3219.54,0.036585,6.77,2.664535e-15,0.0,6.77,6.77,1.0
3997,C03104998,293.0,67.0,49.0,182.0,2.742424,6.023466,-2145.57,-1042.87,861.26,...,2.865920,-450.00,1000.00,0.156997,7.72,0.000000e+00,0.0,7.72,7.72,1.0
3998,C03104999,2378.0,448.0,386.0,710.0,1.586130,0.932191,-39743.75,-563.75,7964.59,...,3.774079,-9999.99,10318.24,0.139613,308.00,0.000000e+00,0.0,308.00,308.00,1.0


In [70]:
final_all_features = all_features_test.fillna(-1000)

In [72]:
# Step 2: Prepare data
X = all_features.drop(columns=['masked_consumer_id', 'FPF_TARGET'])
y = all_features['FPF_TARGET']
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.3, random_state=42)

# Step 3: Train XGBoost for feature learning
pos = sum(y_train == 1)
neg = sum(y_train == 0)
scale_pos_weight = neg / pos

xgb_model = xgb.XGBClassifier(
    use_label_encoder=False,
    eval_metric='auc',
    scale_pos_weight=scale_pos_weight,
    n_estimators=100,
    max_depth=4,
    learning_rate=0.05,
    random_state=42
)
xgb_model.fit(X_train, y_train)

# Step 4: Use XGBoost's leaf output as features
X_train_transformed = xgb_model.apply(X_train)
X_test_transformed = xgb_model.apply(X_test)
print(f"Initial XGBOOST AUC: {roc_auc_score(y_test, xgb_model.predict_proba(X_test)[:,1]):.4f}")

# Optionally standardize (leaf indices can be large integers)
def test(train, test):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(train)
    X_test_scaled = scaler.transform(test)
    log_reg = LogisticRegression(max_iter=1000)
    log_reg.fit(X_train_scaled, y_train)
    y_pred_probs = log_reg.predict_proba(X_test_scaled)[:, 1]
    auc = roc_auc_score(y_test, y_pred_probs)
    print(f"Final AUC with Scale (XGBoost + Logistic Regression): {auc:.4f}")

test(X_train_transformed, X_test_transformed)

# Step 5: Logistic Regression on XGBoost-transformed features
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_transformed, y_train)
y_pred_probs = log_reg.predict_proba(X_test_transformed)[:, 1]

# Step 6: Evaluate final AUC
auc = roc_auc_score(y_test, y_pred_probs)
print(f"Final AUC without Scale (XGBoost + Logistic Regression): {auc:.4f}")


Initial XGBOOST AUC: 0.7459
Final AUC with Scale (XGBoost + Logistic Regression): 0.7011
Final AUC without Scale (XGBoost + Logistic Regression): 0.7041


In [110]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

# Step 2: Prepare data
X = test_data_4.drop(columns=['masked_consumer_id', 'FPF_TARGET'])
y = test_data_4['FPF_TARGET']
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.3, random_state=42)
X_train = X_train.fillna(0)
X_test = X_test.fillna(0)
# Step 3: Random Forest Model
rf_model = RandomForestClassifier(n_estimators=100, max_depth=1000, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_probs_rf = rf_model.predict_proba(X_test)[:, 1]
auc_rf = roc_auc_score(y_test, y_pred_probs_rf)
print(f"AUC (Random Forest): {auc_rf:.4f}")

AUC (Random Forest): 0.6950


In [111]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

n_estimators_list = [20, 50, 100, 500, 1000]
max_depth_list = [5, 10, 20, 50, 100, 250, 500, 1000, None]

for n in n_estimators_list:
    for d in max_depth_list:
        rf_model = RandomForestClassifier(n_estimators=n, max_depth=d, random_state=42)
        rf_model.fit(X_train, y_train)
        y_pred_probs_rf = rf_model.predict_proba(X_test)[:, 1]
        auc_rf = roc_auc_score(y_test, y_pred_probs_rf)
        print(f"AUC (n_estimators={n}, max_depth={d}): {auc_rf:.4f}")


AUC (n_estimators=20, max_depth=5): 0.6672
AUC (n_estimators=20, max_depth=10): 0.6873
AUC (n_estimators=20, max_depth=20): 0.6657
AUC (n_estimators=20, max_depth=50): 0.6349
AUC (n_estimators=20, max_depth=100): 0.6349
AUC (n_estimators=20, max_depth=250): 0.6349
AUC (n_estimators=20, max_depth=500): 0.6349
AUC (n_estimators=20, max_depth=1000): 0.6349
AUC (n_estimators=20, max_depth=None): 0.6349
AUC (n_estimators=50, max_depth=5): 0.6862
AUC (n_estimators=50, max_depth=10): 0.7213
AUC (n_estimators=50, max_depth=20): 0.6918
AUC (n_estimators=50, max_depth=50): 0.6857
AUC (n_estimators=50, max_depth=100): 0.6857
AUC (n_estimators=50, max_depth=250): 0.6857
AUC (n_estimators=50, max_depth=500): 0.6857
AUC (n_estimators=50, max_depth=1000): 0.6857
AUC (n_estimators=50, max_depth=None): 0.6857
AUC (n_estimators=100, max_depth=5): 0.7066
AUC (n_estimators=100, max_depth=10): 0.7446
AUC (n_estimators=100, max_depth=20): 0.6948
AUC (n_estimators=100, max_depth=50): 0.6950
AUC (n_estimators

KeyboardInterrupt: 

In [76]:
8*9*9

648

In [79]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

n_estimators_list = [5, 10, 15, 20, 50, 100, 500, 1000]
max_depth_1 = [5, 10, 20, 50, 100, 250, 500, 1000, None]
lr = [.00001, .0001, .001, .01, .1, .5]
max_depth_2 = [5, 10, 20, 50, 100, 250, 500, 1000, None]
best = 0
saved1, saved2, saved3 = 0, 0, 0
# Step 2: Prepare data
X = all_features.drop(columns=['masked_consumer_id', 'FPF_TARGET'])
y = all_features['FPF_TARGET']
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.3, random_state=42)

# Step 3: Train XGBoost for feature learning
pos = sum(y_train == 1)
neg = sum(y_train == 0)
scale_pos_weight = neg / pos

for i in n_estimators_list:
    for j in max_depth_1:
        for l in lr:
            xgb_model = xgb.XGBClassifier(
                use_label_encoder=False,
                eval_metric='auc',
                scale_pos_weight=scale_pos_weight,
                n_estimators=i,
                max_depth=j,
                learning_rate=l,
                random_state=42
            )
            xgb_model.fit(X_train, y_train)
    
            # Step 4: Use XGBoost's leaf output as features
            X_train_transformed = xgb_model.apply(X_train)
            X_test_transformed = xgb_model.apply(X_test)
            roc_auc = roc_auc_score(y_test, xgb_model.predict_proba(X_test)[:,1])
            if roc_auc > best:
                best = roc_auc
                saved1, saved2, saved3 = i, j, l
                print(f"Initial XGBOOST AUC {i}, {j}, {l}: {roc_auc:.4f}")


Initial XGBOOST AUC 5, 5, 1e-05: 0.6610
Initial XGBOOST AUC 5, 5, 0.01: 0.6691
Initial XGBOOST AUC 5, 5, 0.1: 0.6987
Initial XGBOOST AUC 5, 5, 0.5: 0.7249
Initial XGBOOST AUC 10, 5, 0.5: 0.7267
Initial XGBOOST AUC 15, 5, 0.1: 0.7268
Initial XGBOOST AUC 15, None, 0.5: 0.7320
Initial XGBOOST AUC 20, 5, 0.1: 0.7382
Initial XGBOOST AUC 50, 5, 0.1: 0.7492
Initial XGBOOST AUC 500, 5, 0.01: 0.7509


KeyboardInterrupt: 

In [83]:
all_features

,masked_consumer_id,net_week_series_trend,count_week_series_trend,inflow_week_series_trend,outflow_week_series_trend,week_series_category_0_trend,week_series_category_1_trend,week_series_category_2_trend,week_series_category_3_trend,week_series_category_4_trend,...,34_balance_min,34_balance_max,34_balance_25p,34_balance_75p,35_amount_min,35_balance_std,35_freq_per_month,35_freq_per_week,35_amt_per_weekday,35_amt_per_month
0,C03100001,2.532763,0.080820,-3.679276,1.707118,0.000000,8.754895,-169.171644,10.149108,-4.871593,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,C03100002,10.013128,2.000000,75.295939,-65.282811,0.000000,44.093005,17.326267,0.000000,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,C03100003,51.292377,0.095588,375.922794,-324.630417,-253.383459,65.739889,-49.195471,-20.221083,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,C03100004,-2.696664,0.695565,29.113796,-31.810460,0.000000,2.145596,-56.506849,0.000000,-2.849789,...,NaN,NaN,NaN,NaN,-26.56,NaN,1.0,1.0,-26.56,-26.56
4,C03100005,-0.274092,0.072776,-4.213670,2.299438,19.691781,-2.929210,-2.091014,0.000000,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3995,C03104996,0.966498,-0.035363,-6.784656,-2.003766,-4.998628,0.077659,4.218563,0.000000,-2.380886,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3996,C03104997,-482.478000,-1.200000,0.000000,583.384000,0.000000,0.000000,0.000000,0.000000,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3997,C03104998,-1.797585,0.478844,5.803529,-18.176656,0.000000,0.792059,-1.172464,0.000000,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3998,C03104999,0.092953,0.070250,2.356800,-2.525265,-3.288036,0.644370,2.259434,0.000000,-0.738133,...,308.0,308.0,308.0,308.0,NaN,NaN,NaN,NaN,NaN,NaN


In [105]:
test_data_1 = pd.merge(features_df, trend_df, on = 'masked_consumer_id').merge(fin_table[['masked_consumer_id', 'FPF_TARGET']], on = 'masked_consumer_id') 
test_data_2 = pd.merge(features_df, fin_features, on = 'masked_consumer_id')
test_data_3 = pd.merge(trend_df, fin_features, on = 'masked_consumer_id')
test_data_4 = pd.merge(test_data_1, fin_features.drop(columns = ['FPF_TARGET']), on = 'masked_consumer_id')
features_df_alone = pd.merge(features_df,fin_table[['masked_consumer_id', 'FPF_TARGET']], on = 'masked_consumer_id') 
trend_df_alone = pd.merge(trend_df,fin_table[['masked_consumer_id', 'FPF_TARGET']], on = 'masked_consumer_id') 
ct = 0
for dataset in [test_data_1, test_data_2, test_data_3, test_data_4, features_df_alone, trend_df_alone, all_features]:

    if 'FPF_TARGET' not in list(dataset.columns):
        print(ct)
    ct += 1

In [95]:
test_data_4.col

,masked_consumer_id,n_records,n_unique_posted_days,multi_entry_days,active_days,mean_gap_days,std_gap_days,dow_0_total_spent,dow_1_total_spent,dow_2_total_spent,...,34_balance_max,34_balance_25p,34_balance_75p,35_amount_min,35_balance_std,35_freq_per_month,35_freq_per_week,35_amt_per_weekday,35_amt_per_month,FPF_TARGET
0,C03100001,2511.0,368.0,354.0,535.0,1.455041,0.881243,-42149.46,-11659.15,-1578.09,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
1,C03100002,1252.0,133.0,129.0,197.0,1.484848,0.891754,-11862.39,1322.98,-361.53,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
2,C03100003,445.0,104.0,93.0,120.0,1.155340,0.516848,21093.80,-3004.72,-13205.08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
3,C03100004,1326.0,160.0,157.0,228.0,1.427673,0.864820,-1940.55,1196.45,-2608.91,...,NaN,NaN,NaN,-26.56,NaN,1.0,1.0,-26.56,-26.56,0.0
4,C03100005,772.0,299.0,197.0,480.0,1.607383,0.946672,-1284.88,-1410.02,-872.77,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3995,C03104996,1675.0,339.0,296.0,526.0,1.553254,0.947656,-17135.57,-14376.23,32254.67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
3996,C03104997,82.0,18.0,15.0,30.0,1.705882,1.225451,-1206.78,-257.03,-330.80,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
3997,C03104998,293.0,67.0,49.0,182.0,2.742424,6.023466,-2145.57,-1042.87,861.26,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
3998,C03104999,2378.0,448.0,386.0,710.0,1.586130,0.932191,-39743.75,-563.75,7964.59,...,308.0,308.0,308.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0


In [108]:
"FPF_TARGET" in fin_features.columns

True

In [106]:
# Step 2: Prepare data
for dataset in [test_data_1, test_data_2, test_data_3, test_data_4, features_df_alone, trend_df_alone, all_features]:
    X = dataset.drop(columns=['masked_consumer_id', 'FPF_TARGET'])
    y = dataset['FPF_TARGET']
    X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.3, random_state=42)
    
    # Step 3: Train XGBoost for feature learning
    pos = sum(y_train == 1)
    neg = sum(y_train == 0)
    scale_pos_weight = neg / pos
    
    xgb_model = xgb.XGBClassifier(
        use_label_encoder=False,
        eval_metric='auc',
        scale_pos_weight=scale_pos_weight,
        n_estimators=500,
        max_depth=5,
        learning_rate=0.01,
        random_state=42
    )
    xgb_model.fit(X_train, y_train)

    # Step 4: Use XGBoost's leaf output as features
    X_train_transformed = xgb_model.apply(X_train)
    X_test_transformed = xgb_model.apply(X_test)
    print(f"Initial XGBOOST AUC: {roc_auc_score(y_test, xgb_model.predict_proba(X_test)[:,1]):.4f}")
    
    # Optionally standardize (leaf indices can be large integers)
    def test(train, test):
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(train)
        X_test_scaled = scaler.transform(test)
        log_reg = LogisticRegression(max_iter=1000)
        log_reg.fit(X_train_scaled, y_train)
        y_pred_probs = log_reg.predict_proba(X_test_scaled)[:, 1]
        auc = roc_auc_score(y_test, y_pred_probs)
        print(f"Final AUC with Scale (XGBoost + Logistic Regression): {auc:.4f}")
    
    test(X_train_transformed, X_test_transformed)
    
    # Step 5: Logistic Regression on XGBoost-transformed features
    log_reg = LogisticRegression(max_iter=1000)
    log_reg.fit(X_train_transformed, y_train)
    y_pred_probs = log_reg.predict_proba(X_test_transformed)[:, 1]
    
    # Step 6: Evaluate final AUC
    auc = roc_auc_score(y_test, y_pred_probs)
    print(f"Final AUC without Scale (XGBoost + Logistic Regression): {auc:.4f}")


Initial XGBOOST AUC: 0.6939
Final AUC with Scale (XGBoost + Logistic Regression): 0.6157
Final AUC without Scale (XGBoost + Logistic Regression): 0.6081
Initial XGBOOST AUC: 0.7474
Final AUC with Scale (XGBoost + Logistic Regression): 0.6973
Final AUC without Scale (XGBoost + Logistic Regression): 0.6472
Initial XGBOOST AUC: 0.7509
Final AUC with Scale (XGBoost + Logistic Regression): 0.7125
Final AUC without Scale (XGBoost + Logistic Regression): 0.7205
Initial XGBOOST AUC: 0.7480
Final AUC with Scale (XGBoost + Logistic Regression): 0.7079
Final AUC without Scale (XGBoost + Logistic Regression): 0.6950
Initial XGBOOST AUC: 0.6684
Final AUC with Scale (XGBoost + Logistic Regression): 0.6434
Final AUC without Scale (XGBoost + Logistic Regression): 0.6602
Initial XGBOOST AUC: 0.6637
Final AUC with Scale (XGBoost + Logistic Regression): 0.6208
Final AUC without Scale (XGBoost + Logistic Regression): 0.6268
Initial XGBOOST AUC: 0.7509
Final AUC with Scale (XGBoost + Logistic Regression): 

In [109]:
X = fin_features.drop(columns=['masked_consumer_id', 'FPF_TARGET'])
y = fin_features['FPF_TARGET']
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.3, random_state=42)

# Step 3: Train XGBoost for feature learning
pos = sum(y_train == 1)
neg = sum(y_train == 0)
scale_pos_weight = neg / pos

xgb_model = xgb.XGBClassifier(
    use_label_encoder=False,
    eval_metric='auc',
    scale_pos_weight=scale_pos_weight,
    n_estimators=500,
    max_depth=5,
    learning_rate=0.01,
    random_state=42
)
xgb_model.fit(X_train, y_train)

# Step 4: Use XGBoost's leaf output as features
X_train_transformed = xgb_model.apply(X_train)
X_test_transformed = xgb_model.apply(X_test)
print(f"Initial XGBOOST AUC: {roc_auc_score(y_test, xgb_model.predict_proba(X_test)[:,1]):.4f}")

# Optionally standardize (leaf indices can be large integers)
def test(train, test):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(train)
    X_test_scaled = scaler.transform(test)
    log_reg = LogisticRegression(max_iter=1000)
    log_reg.fit(X_train_scaled, y_train)
    y_pred_probs = log_reg.predict_proba(X_test_scaled)[:, 1]
    auc = roc_auc_score(y_test, y_pred_probs)
    print(f"Final AUC with Scale (XGBoost + Logistic Regression): {auc:.4f}")

test(X_train_transformed, X_test_transformed)
# Step 5: Logistic Regression on XGBoost-transformed features
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_transformed, y_train)
y_pred_probs = log_reg.predict_proba(X_test_transformed)[:, 1]

# Step 6: Evaluate final AUC
auc = roc_auc_score(y_test, y_pred_probs)
print(f"Final AUC without Scale (XGBoost + Logistic Regression): {auc:.4f}")


Initial XGBOOST AUC: 0.7484
Final AUC with Scale (XGBoost + Logistic Regression): 0.7203
Final AUC without Scale (XGBoost + Logistic Regression): 0.6632


In [99]:
for i in X.columns:
    if "fpf" in i.lower():
        print(i)
#f"FPF_TARGET"X.columns

FPF_TARGET_x
FPF_TARGET_y
